# AIC 2026 — OCR keyframe bằng **baidu/Unlimited-OCR** trên Kaggle (GPU T4 x2)

Model: [`baidu/Unlimited-OCR`](https://huggingface.co/baidu/Unlimited-OCR) — VLM ~3B (DeepEncoder: SAM-ViT-B + CLIP-L → DeepSeek-V2 MoE), MIT license, bản đi tiếp từ DeepSeek-OCR.

**Khác gì notebook cũ (PaddleOCR-det + VietOCR)?**
- Bỏ hẳn `paddlepaddle` / `paddleocr` / `vietocr` → không còn rắc rối CUDA-paddle.
- Model làm **detection + recognition + layout trong một lượt forward**, output là Markdown kèm thẻ toạ độ `<|ref|>…<|/ref|><|det|>[[x1,y1,x2,y2]]<|/det|>` (toạ độ chuẩn hoá theo thang **0–999**).
- API remote code là `model.infer(...)`, **1 ảnh / 1 lần gọi** (không có batch API).

---

### ⚙️ Settings Kaggle (panel bên phải)
- **Accelerator** → `GPU T4 x2`
- **Internet** → `ON` (cần `pip install` + tải ~6.7 GB weight từ HuggingFace)

### 🔧 Ba thứ phải vá để model chạy được trên T4 — đọc kỹ

**1. dtype: PHẢI dùng `bf16`, KHÔNG được dùng `fp16`.** Remote code hardcode `.bfloat16()` và `torch.autocast("cuda", dtype=torch.bfloat16)`, không có cờ nào để đổi. T4 là **sm_75, không có hardware bf16**, nên phản xạ đầu tiên là ép sang fp16 (`torch.bfloat16 = torch.float16` — remote code tra attribute này lúc *runtime* nên gán lại là đủ).

> 🔴 **Đã chạy thật trên T4 x2 và fp16 KHÔNG DÙNG ĐƯỢC.** Model tràn NaN → logits `NaN` → `argmax` luôn trả token 0 → output chỉ là `<｜begin▁of▁sentence｜>` lặp lại vô hạn, `text` rỗng 100%. Nó **không báo lỗi gì**, vẫn chạy 2.1 s/ảnh như thường, nên rất dễ tưởng là đang chạy tốt.
>
> → Đặt `DTYPE_MODE = 'bf16'`. PyTorch 2.10 **emulate được bf16 trên sm_75**: chậm hơn fp16 nhưng số học đúng. Nếu bf16 cũng NaN thì đổi Accelerator sang **`GPU L4 x4`** (sm_89, bf16 native — vừa đúng vừa nhanh hơn T4).
>
> Notebook có hàm `is_degenerate()` bắt đúng dạng rác này, và ô chạy full có `assert` chặn không cho chạy nếu smoke test bị degenerate.

**2. Bắt buộc `attn_implementation='eager'`.** `config.use_mla = False` nên mỗi decoder layer tra key `"mha_" + attn_implementation` trong `ATTENTION_CLASSES` của `modeling_deepseekv2.py`, mà bảng đó **chỉ có `mha_eager`** — `mha_flash_attention_2` bị comment out và không có key `sdpa` nào. Để `transformers` tự chọn (nó thích `sdpa`) sẽ vỡ với `KeyError: 'mha_sdpa'`. **Vì vậy đừng cài `flash-attn`** — vô ích với model này.

**3. `output_path` không được để rỗng.** `infer()` gọi `os.makedirs(output_path)` **vô điều kiện**, không quan tâm `save_results`. Truyền `output_path=''` → mọi ảnh chết với `FileNotFoundError`.

### 🖥️ Dùng cả 2 GPU thế nào?
Weight fp16 ~6.7 GB → **vừa gọn trong 1 GPU T4 16 GB**, nên chia model ra 2 GPU là vô nghĩa (chỉ thêm overhead copy). Cách dùng 2 GPU đúng ở đây là **2 worker song song, mỗi GPU giữ 1 bản model riêng**, mỗi worker OCR một nửa số keyframe.
> Lưu ý thực tế: `generate()` của model MoE này nặng phần Python, nên 2 thread sẽ **tranh GIL** → tăng tốc thường chỉ ~1.3–1.6x, không phải 2x. Vẫn đáng, nhưng đừng kỳ vọng gấp đôi.

In [ ]:
# ============================================================
# CAI DAT — chay o nay DAU TIEN trong kernel con "sach",
# roi Run -> Restart Session, sau do chay tiep tu o Cau hinh.
# ============================================================
import sys, subprocess

def sh(cmd, check=True):
    print('>>', cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=check)

PIP = f'{sys.executable} -m pip install -q'

# 1) Remote code viet cho transformers 4.5x (config ghi 4.46.3, model card test 4.57.1).
#    transformers 5.x lam vo remote code -> PIN CHAT.
#    Khong pin tokenizers, de pip tu resolve ban khop.
sh(f'{PIP} "transformers==4.57.1" "huggingface_hub>=0.30"')

# 2) Cac dependency ma remote code import truc tiep
sh(f'{PIP} einops==0.8.2 addict==2.4.0 easydict==1.13 psutil matplotlib')

# 3) pymupdf CHI can neu OCR file PDF. Keyframe la anh -> khong bat buoc.
sh(f'{PIP} pymupdf')

# 4) KHONG cai lai torch/torchvision (torch cua Kaggle chay tot, cai lai keo ~3GB va de lech CUDA).
# 5) KHONG cai flash-attn: use_mla=False -> can key "mha_flash_attention_2",
#    ma key do bi comment out trong ATTENTION_CLASSES. Model chi chay voi 'eager'.

import torch, transformers
print('transformers :', transformers.__version__)
print('torch        :', torch.__version__, '| cuda', torch.version.cuda)
n_gpu = torch.cuda.device_count()
print('so GPU thay duoc:', n_gpu)
for i in range(n_gpu):
    cap = torch.cuda.get_device_capability(i)
    props = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i} {props.name} sm_{cap[0]}{cap[1]} | {props.total_memory/1e9:.1f} GB')
if n_gpu == 0:
    print('!! KHONG CO GPU -> model nay khong chay duoc (remote code hardcode .cuda())')

## Cấu hình Dataset & tham số chạy

In [ ]:
from pathlib import Path
import torch

# 1. Dataset mount o /kaggle/input/<dataset-slug>  (KHONG co phan "datasets/<username>/")
#    Chay  !ls /kaggle/input  neu khong chac ten.
DATASET_DIRECTORY = Path('/kaggle/input/aic-dataset')

# 2. Thu muc output
DEMO_OUTPUT = Path('/kaggle/working/ocr_unlimited')

# 3. Folder keyframe chay cho session nay
TARGET_FOLDER = 'Keyframes_L22'

MODEL_PATH = 'baidu/Unlimited-OCR'

# 4. Dung GPU nao. T4 x2 -> [0, 1]. Dat [0] neu muon chi 1 GPU (de debug cho de).
GPU_IDS = list(range(torch.cuda.device_count())) or [0]

# 5. DTYPE. Doc ky - day la cho DE VO NHAT tren T4.
#      'bf16' -> dung dtype goc model duoc train. T4 (sm_75) khong co hardware bf16 nhung
#                PyTorch 2.10 emulate duoc -> CHAM hon, nhung SO HOC DUNG. Nen thu cai nay TRUOC.
#      'fp16' -> nhanh tren T4 (co tensor core fp16) NHUNG range hep hon bf16 rat nhieu.
#                Da test: model nay TRAN NaN voi fp16 -> logits NaN -> argmax luon ra token 0
#                -> output chi la '<|begin_of_sentence|>' lap vo han. Coi nhu khong dung duoc.
#      'auto' -> tu do xem GPU chay noi bf16 khong, uu tien bf16 (dung > nhanh).
DTYPE_MODE = 'bf16'

# 6. Che do resolution, theo dung model card:
#      'gundam' -> base_size=1024, image_size=640, crop_mode=True
#                  tile anh thanh nhieu patch -> net hon voi chu nho, NHUNG anh 16:9
#                  bi tile thanh (7,4) = 28 tile -> ~3113 image token, CHAM han.
#      'base'   -> image_size=1024, crop_mode=False
#                  1 luot xem toan anh -> 273 image token, NHANH, du cho keyframe video.
RES_MODE = 'base'

RES_CONFIG = {
    'gundam': dict(base_size=1024, image_size=640,  crop_mode=True),
    'base':   dict(base_size=1024, image_size=1024, crop_mode=False),
}
assert RES_MODE in RES_CONFIG, f'RES_MODE khong hop le: {RES_MODE}'

# 7. Tham so sinh chuoi.
#    !! MAX_LENGTH la max_length cua generate() = TONG chieu dai chuoi (prompt + output),
#       KHONG phai max_new_tokens. Prompt da chua rat nhieu image token:
#         RES_MODE 'base'   -> 273 image token
#         RES_MODE 'gundam' -> anh 16:9 tile (7,4) -> ~3113 image token
#       Neu MAX_LENGTH <= do dai prompt, generate() dung ngay va tra ve chuoi RONG
#       (chi log warning, KHONG raise) -> rat de tuong la model doc khong ra chu.
#       4096 an toan cho ca hai che do. Model card dung 32768.
MAX_LENGTH = 4096
NO_REPEAT_NGRAM_SIZE = 35   # theo model card, chong lap vo han
NGRAM_WINDOW = 128          # 128 cho anh don, 1024 cho multi-page
TEMPERATURE = 0.0           # greedy -> deterministic

PROMPT = '<image>document parsing.'   # prompt duy nhat model card document cho anh don
MODEL_ID = f'Unlimited-OCR ({RES_MODE})'

CHECKPOINT_EVERY = 50       # flush checkpoint moi N anh (phong session Kaggle bi ngat)
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}

# 8. infer() goi os.makedirs(output_path) VO DIEU KIEN, bat ke save_results.
#    output_path='' -> FileNotFoundError cho MOI anh. Phai la thu muc that.
INFER_WORKDIR = DEMO_OUTPUT / '_infer_tmp'

DEMO_OUTPUT.mkdir(parents=True, exist_ok=True)
INFER_WORKDIR.mkdir(parents=True, exist_ok=True)

if not DATASET_DIRECTORY.is_dir():
    mounted = sorted(p.name for p in Path('/kaggle/input').iterdir()) if Path('/kaggle/input').is_dir() else []
    raise AssertionError(f'Khong thay dataset: {DATASET_DIRECTORY}. /kaggle/input dang co: {mounted}')
target_path = DATASET_DIRECTORY / TARGET_FOLDER
assert target_path.is_dir(), (f'Khong thay thu muc con {TARGET_FOLDER}. '
                              f'Dataset dang co: {sorted(p.name for p in DATASET_DIRECTORY.iterdir())[:20]}')

keyframe_roots = [target_path]
print(f'Folder     : {TARGET_FOLDER}')
print(f'res_mode   : {RES_MODE} | {RES_CONFIG[RES_MODE]}')
print(f'GPU dung   : {GPU_IDS}')
print(f'max_length : {MAX_LENGTH} (tong prompt+output)')

## Liệt kê toàn bộ keyframe trong target folder

In [ ]:
import re

VIDEO_ID_PATTERN = re.compile(r'^L\d{2}_V\d{3}$')

def find_video_dirs(root):
    base = root / 'keyframes'
    if not base.is_dir():
        base = root
    return [p for p in sorted(base.iterdir())
            if p.is_dir() and VIDEO_ID_PATTERN.match(p.name)]

def list_images(video_dir):
    return sorted((p for p in video_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS),
                  key=lambda p: (not p.stem.isdigit(), int(p.stem) if p.stem.isdigit() else 0, p.name))

all_video_dirs = [v for root in keyframe_roots for v in find_video_dirs(root)]
all_video_dirs.sort(key=lambda p: p.name)
assert all_video_dirs, 'Khong tim thay video nao'

samples = []  # [(video_dir, [image_path, ...])]
for video_dir in all_video_dirs:
    images = list_images(video_dir)
    if images:
        samples.append((video_dir, images))

total_keyframes = sum(len(imgs) for _, imgs in samples)
print(f'{len(samples)} video | tong {total_keyframes} keyframe')
for video_dir, images in samples[:20]:
    print(f'- {video_dir.name}: {len(images)} keyframe')
if len(samples) > 20:
    print(f'... va {len(samples) - 20} video nua')

## Tải Unlimited-OCR (1 bản / GPU)

Lần đầu tải ~6.7 GB weight về `~/.cache/huggingface` (cần Internet = ON) — mất vài phút. Bản thứ hai load từ cache nên nhanh hơn.

`trust_remote_code=True` là **bắt buộc**: kiến trúc `UnlimitedOCRForCausalLM` không có trong `transformers`, nó được nạp từ `modeling_unlimitedocr.py` trong repo.

In [ ]:
import time, warnings, torch, transformers
from transformers import AutoModel, AutoTokenizer

assert torch.cuda.is_available(), 'Remote code hardcode .cuda() -> phai bat GPU'

# Dep log rac: moi lan generate(), transformers in lai "attention mask ... not set"
# va "Setting pad_token_id" -> chay 9096 anh thi log Kaggle khong doc duoc gi.
transformers.logging.set_verbosity_error()
warnings.filterwarnings('ignore')

# ---------- Chon dtype ----------
# Neu torch._bf16_orig ton tai nghia la kernel nay tung bi va fp16 -> tra lai bf16 that truoc.
if getattr(torch, '_bf16_orig', None) is not None:
    torch.bfloat16 = torch._bf16_orig

def bf16_works():
    """T4 khong co hardware bf16, nhung PyTorch co the emulate. Thu 1 phep matmul that."""
    try:
        a = torch.randn(64, 64, device=f'cuda:{GPU_IDS[0]}', dtype=torch.bfloat16)
        return not torch.isnan(a @ a).any().item()
    except Exception as e:
        print('  bf16 matmul that bai:', type(e).__name__, str(e)[:120])
        return False

mode = DTYPE_MODE
if mode == 'auto':
    mode = 'bf16' if bf16_works() else 'fp16'
    print(f'DTYPE_MODE=auto -> chon {mode}')

if mode == 'bf16':
    assert bf16_works(), 'GPU nay khong chay noi bf16 -> dat DTYPE_MODE = "fp16"'
    DTYPE = torch.bfloat16
    NEED_FP16_PATCH = False
    cap0 = torch.cuda.get_device_capability(GPU_IDS[0])
    print(f'dtype = bfloat16 (dtype goc cua model)')
    if cap0[0] < 8:
        print(f'   luu y: sm_{cap0[0]}{cap0[1]} khong co hardware bf16, PyTorch dang emulate '
              f'-> cham hon fp16 nhung SO HOC DUNG.')
elif mode == 'fp16':
    # ---------- VA 1: bfloat16 -> float16 ----------
    # Remote code hardcode .bfloat16() va torch.autocast(dtype=torch.bfloat16), khong co co de doi.
    # No tra torch.bfloat16 luc RUNTIME nen gan lai attribute nay la du de doi toan bo sang fp16.
    if getattr(torch, '_bf16_orig', None) is None:
        torch._bf16_orig = torch.bfloat16
    torch.bfloat16 = torch.float16          # <-- ban va
    DTYPE = torch.float16
    NEED_FP16_PATCH = True
    print('dtype = float16 (da va torch.bfloat16 := torch.float16)')
    print('   !! DA TEST: model nay TRAN NaN o fp16 -> output chi la <|begin_of_sentence|> lap lai.')
    print('   !! Neu smoke test bao "degenerate" thi doi DTYPE_MODE = "bf16".')
else:
    raise ValueError(f'DTYPE_MODE khong hop le: {DTYPE_MODE}')

t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

def load_model_on(gpu_id):
    """Load 1 ban model len cuda:<gpu_id>."""
    with torch.cuda.device(gpu_id):
        m = AutoModel.from_pretrained(
            MODEL_PATH,
            trust_remote_code=True,
            use_safetensors=True,
            torch_dtype=DTYPE,          # transformers 4.x; v5 doi ten thanh dtype
            # ---------- VA 2: BAT BUOC 'eager' ----------
            # use_mla=False -> DecoderLayer tra key "mha_" + _attn_implementation,
            # ma ATTENTION_CLASSES chi co 'mha_eager'.
            # De transformers tu chon -> KeyError: 'mha_sdpa'
            # Ep 'flash_attention_2'  -> KeyError: 'mha_flash_attention_2'
            attn_implementation='eager',
        )
        m = m.eval().to(f'cuda:{gpu_id}').to(DTYPE)
    return m

models = {}
for gid in GPU_IDS:
    models[gid] = load_model_on(gid)
    print(f'  cuda:{gid} da nap xong | VRAM {torch.cuda.memory_allocated(gid)/1e9:.1f} GB')

load_seconds = time.perf_counter() - t0
n_params = sum(p.numel() for p in models[GPU_IDS[0]].parameters()) / 1e9
print(f'\nLoad {len(models)} ban model: {load_seconds:.1f}s | {n_params:.2f}B params/ban | {MODEL_ID}')
print('attn_implementation =', getattr(models[GPU_IDS[0]].config, '_attn_implementation', '?'))

## Hàm OCR + làm sạch output

`model.infer(...)` trả về **string** khi `eval_mode=True`. Mặc định (`eval_mode=False`) nó *stream ra stdout* và trả về `None` — chạy vài nghìn ảnh như vậy thì log Kaggle sẽ nổ, nên bắt buộc bật `eval_mode=True`.

Output thô có 2 dạng thẻ (xem `re_match()` trong `modeling_unlimitedocr.py`):
- `<|ref|>NỘI DUNG<|/ref|><|det|>[[x1,y1,x2,y2]]<|/det|>` — chữ nằm **trong** `<|ref|>`
- `<|det|>label [box]<|/det|>NỘI DUNG` — chữ nằm **sau** thẻ

`to_plain_text` bóc phần chữ ra để nhét vào index retrieval; bản thô vẫn giữ ở field `raw` nên không mất thông tin (kể cả toạ độ).

In [ ]:
import contextlib, io, re

# Bo <|det|>...<|/det|> truoc, roi bo cac tag con lai -> con lai dung phan chu.
DET_BLOCK   = re.compile(r'<\|det\|>[^<]*?<\|/det\|>')
# CHU Y: model dung CA pipe ASCII '|' VA pipe fullwidth U+FF5C, vd '<｜end▁of▁sentence｜>'.
# infer() chi strip stop_str o CUOI chuoi nen tag fullwidth con sot giua chuoi -> phai bat ca hai.
SPECIAL_TAG = re.compile('<[|｜][^<>]{0,60}[|｜]>')
HTML_TAG    = re.compile(r'</?[a-zA-Z][^<>]{0,80}>')          # <td>, <table>, <br>...
MD_IMAGE    = re.compile(r'!\[[^\]]*\]\([^)]*\)')
COORD_RUN   = re.compile(r'\[\[?\s*\d+\s*(?:,\s*\d+\s*)+\]?\]')
MD_NOISE    = re.compile('[#*_`>|｜▁]+')

# Dau hieu logits NaN: argmax luon ra token 0 = bos -> output la bos lap lai.
BOS_RUN = re.compile('(<[|｜][^<>]{0,40}begin[^<>]{0,40}[|｜]>){3,}')

def is_degenerate(raw):
    """True neu output la rac kieu NaN (bos/pad lap lai, khong co chu nao)."""
    if not raw:
        return False
    if BOS_RUN.search(raw):
        return True
    stripped = SPECIAL_TAG.sub('', raw).strip()
    # >=6 special token ma khong con chu nao -> gan nhu chac la NaN
    return len(stripped) == 0 and len(SPECIAL_TAG.findall(raw)) >= 6

def to_plain_text(raw):
    """Boc phan chu thuan tu output Markdown + tag <|det|> cua Unlimited-OCR."""
    if not raw:
        return ''
    s = DET_BLOCK.sub(' ', raw)
    s = MD_IMAGE.sub(' ', s)
    s = SPECIAL_TAG.sub(' ', s)
    s = COORD_RUN.sub(' ', s)
    s = HTML_TAG.sub(' ', s)
    s = MD_NOISE.sub(' ', s)
    return re.sub(r'\s+', ' ', s).strip()

@torch.inference_mode()
def ocr_one(image_path, gpu_id=None):
    """OCR 1 keyframe -> dict(raw, text, total, gpu). Loi -> text rong + field error."""
    gpu_id = GPU_IDS[0] if gpu_id is None else gpu_id
    cfg = RES_CONFIG[RES_MODE]
    t = time.perf_counter()
    err, raw = None, ''
    try:
        # .cuda() trong remote code dung CURRENT device (thread-local) -> phai set context.
        with torch.cuda.device(gpu_id):
            # bit stdout: remote code co vai lenh print noi bo
            buf = io.StringIO()
            with contextlib.redirect_stdout(buf):
                raw = models[gpu_id].infer(
                    tokenizer,
                    prompt=PROMPT,
                    image_file=str(image_path),      # chi nhan duong dan, KHONG nhan PIL.Image
                    # VA 3: infer() chay os.makedirs(output_path) VO DIEU KIEN
                    # -> '' se raise FileNotFoundError cho moi anh.
                    output_path=str(INFER_WORKDIR),
                    base_size=cfg['base_size'],
                    image_size=cfg['image_size'],
                    crop_mode=cfg['crop_mode'],
                    max_length=MAX_LENGTH,
                    no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
                    ngram_window=NGRAM_WINDOW,
                    temperature=TEMPERATURE,
                    save_results=False,              # khong ghi anh/markdown cho tung keyframe
                    eval_mode=True,                  # <-- can co de infer() RETURN string
                )
        if not isinstance(raw, str):
            raw = raw[0] if isinstance(raw, (tuple, list)) and raw else str(raw or '')
    except torch.cuda.OutOfMemoryError as e:
        torch.cuda.empty_cache()
        err = f'OOM: {e}'
    except Exception as e:
        err = f'{type(e).__name__}: {e}'

    out = {'raw': raw or '', 'text': to_plain_text(raw), 'gpu': gpu_id,
           'total': time.perf_counter() - t}
    if err:
        out['error'] = err[:300]
    elif is_degenerate(raw):
        # Bat o day de KHONG ghi 9096 dong rac vao checkpoint roi moi phat hien.
        out['error'] = 'DEGENERATE: output toan bos/pad -> logits NaN (dtype sai?)'
    return out

## 👀 Smoke test + VISUALIZE — xem OCR có tốt không

Ô dưới chạy vài keyframe rồi vẽ ra: **trái** = ảnh gốc + box đánh số, **phải** = chữ model đọc được ứng với từng số. Toạ độ trong output ở thang 0–999 nên phải nhân lại theo kích thước ảnh thật (`x / 999 * width`) — đúng như `draw_bounding_boxes()` trong remote code làm.

Nếu model chỉ trả text thuần (không có thẻ `<|det|>`), panel phải sẽ hiện nguyên đoạn text.

In [ ]:
import ast
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

# matplotlib mac dinh dung DejaVu Sans - co day du dau tieng Viet.

REF_DET   = re.compile(r'<\|ref\|>(.*?)<\|/ref\|>\s*<\|det\|>(.*?)<\|/det\|>', re.DOTALL)
# CHU Y: det_pattern goc trong re_match() dung r'(\[[^\]]+\])' nen KHONG khop duoc
# box long 2 lop '[[x1,y1,x2,y2]]' (no dung o ']' dau tien). Dung [^<>]+ de bat ca 2 dang.
DET_LABEL = re.compile(r'<\|det\|>\s*([A-Za-z_][\w-]*)\s*(\[[^<>]+\])\s*<\|/det\|>')
INT_RUN   = re.compile(r'-?\d+')

def _parse_boxes(s):
    """'[[x1,y1,x2,y2], ...]' hoac '[x1,y1,x2,y2]' -> [[x1,y1,x2,y2], ...] (thang 0-999)."""
    s = s.strip()
    out = []
    try:
        v = ast.literal_eval(s)
        if isinstance(v, (list, tuple)) and v and isinstance(v[0], (int, float)):
            v = [v]
        for b in v if isinstance(v, (list, tuple)) else []:
            if isinstance(b, (list, tuple)) and len(b) >= 4:
                out.append([float(x) for x in b[:4]])
    except Exception:
        out = []
    if not out:
        # fallback: vet moi so nguyen roi chia tung nhom 4
        nums = [float(x) for x in INT_RUN.findall(s)]
        out = [nums[i:i + 4] for i in range(0, len(nums) - 3, 4)]
    return out

def parse_regions(raw):
    """-> [{'label', 'text', 'boxes'}] ; boxes o thang 0-999."""
    regions = []
    for text, box_str in REF_DET.findall(raw or ''):
        boxes = _parse_boxes(box_str)
        if boxes:
            regions.append({'label': 'ref', 'text': to_plain_text(text), 'boxes': boxes})
    if not regions:
        # dang 2: <|det|>label [box]<|/det|> NOI DUNG   (chu nam SAU the)
        hits = list(DET_LABEL.finditer(raw or ''))
        for i, m in enumerate(hits):
            end = hits[i + 1].start() if i + 1 < len(hits) else len(raw)
            boxes = _parse_boxes(m.group(2))
            if boxes:
                regions.append({'label': m.group(1),
                                'text': to_plain_text(raw[m.end():end]),
                                'boxes': boxes})
    return regions

def show_ocr(image_path, res, max_regions=30):
    img = Image.open(image_path).convert('RGB')
    W, H = img.size
    regions = parse_regions(res['raw'])

    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(17, 7.5),
                                  gridspec_kw={'width_ratios': [1.3, 1]})
    ax.imshow(img); ax.axis('off')
    tag = f' | cuda:{res.get("gpu")}' if res.get('gpu') is not None else ''
    ax.set_title(f'{image_path.parent.name}/{image_path.name}  {W}x{H}  '
                 f'{res["total"]*1000:.0f} ms{tag}', fontsize=11)

    lines = []
    for i, r in enumerate(regions[:max_regions], 1):
        for b in r['boxes']:
            x1 = b[0] / 999 * W; y1 = b[1] / 999 * H
            x2 = b[2] / 999 * W; y2 = b[3] / 999 * H
            ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                            fill=False, linewidth=2, edgecolor='#e63946'))
            ax.text(x1, max(0, y1 - 3), str(i), fontsize=9, color='white', va='bottom',
                    bbox=dict(facecolor='#e63946', pad=1.2, edgecolor='none'))
        txt = r['text'] or '(rong)'
        lines.append(f'{i}. [{r["label"]}] {txt[:100]}')

    ax2.axis('off')
    if regions:
        ax2.set_title(f'{len(regions)} vung text model doc duoc', fontsize=11)
        body = '\n'.join(lines)
        if len(regions) > max_regions:
            body += f'\n... va {len(regions) - max_regions} vung nua'
    elif res.get('error'):
        ax2.set_title('LOI', fontsize=11)
        body = res['error']
    else:
        ax2.set_title('Khong co toa do -> text thuan', fontsize=11)
        body = res['text'][:1500] or '(RONG - khong doc duoc chu nao)'
    ax2.text(0, 1, body, va='top', ha='left', fontsize=10, wrap=True)
    plt.tight_layout(); plt.show()


# ---- chay thu: lay 4 keyframe rai rac cho de danh gia ----
flat = [(vd, p) for vd, imgs in samples for p in imgs]
step = max(1, len(flat) // 4)
probe_items = flat[::step][:4]

probe = []
for k, (vd, p) in enumerate(probe_items):
    res = ocr_one(p, gpu_id=GPU_IDS[k % len(GPU_IDS)])
    probe.append(res)
    show_ocr(p, res)
    print('RAW (300 ky tu dau):', (res['raw'][:300] or '(rong)').replace('\n', ' ⏎ '))
    print('TEXT sach          :', res['text'][:200] or '(rong)')
    print('-' * 100)

# ---- chan doan ----
n_degen = sum(1 for r in probe if is_degenerate(r['raw']))
n_err   = sum(1 for r in probe if r.get('error') and not is_degenerate(r['raw']))
n_empty = sum(1 for r in probe if not r['text'] and not r.get('error'))
n_ok    = sum(1 for r in probe if r['text'])
mean_ms = sum(r['total'] for r in probe) / len(probe) * 1000

print(f'\n{len(probe)} anh thu | {n_ok} co chu | {n_degen} DEGENERATE | {n_err} loi | {n_empty} rong')
print(f'{mean_ms:.0f} ms/anh -> uoc tinh {total_keyframes} keyframe: '
      f'~{mean_ms/1000*total_keyframes/3600/len(GPU_IDS):.1f} gio voi {len(GPU_IDS)} GPU '
      f'(Kaggle kill kernel sau 12h)')

if n_degen:
    print(f'\n!! {n_degen}/{len(probe)} anh DEGENERATE: output chi la <|begin_of_sentence|> lap lai.')
    print('   Nghia la logits = NaN -> argmax luon tra token 0. KHONG PHAI loi anh, la loi DTYPE.')
    print(f'   Dang chay dtype = {DTYPE}.')
    if DTYPE == torch.float16:
        print('   -> SUA: dat DTYPE_MODE = "bf16" o o Cau hinh, Restart Session, chay lai.')
    else:
        print('   -> bf16 cung NaN. Doi Accelerator sang GPU L4 x4 (sm_89, bf16 native).')
    print('   DUNG chay o "toan bo dataset" khi con dong nay.')
elif n_err:
    print('\n!! Co loi that (khong phai degenerate). Doc field error o tren.')
elif n_empty == len(probe):
    print('\n!! TAT CA RONG (nhung khong degenerate). Kiem tra theo thu tu:')
    print(f'   1) MAX_LENGTH > so image token? (base=273, gundam~3113) -> dang la {MAX_LENGTH}')
    print('   2) anh co chu that khong; thu RES_MODE = "gundam" cho chu nho')
else:
    print(f'\nOK - {n_ok}/{len(probe)} anh doc ra chu. Xem hinh o tren xem box va chu co dung khong '
          f'roi hay chay o toan bo dataset.')

## Chạy toàn bộ dataset — 2 GPU song song, có checkpoint & resume

Mỗi GPU là một worker riêng với bản model riêng, nhận một nửa danh sách (chia xen kẽ `todo[i::n]` để tải đều nhau). Checkpoint ghi chung 1 file JSONL, có lock.

Kernel Kaggle bị kill sau 12h → nếu session đứt, chỉ cần **chạy lại ô này**, nó tự bỏ qua keyframe đã xong.

In [ ]:
import json, threading

# ---------- CHOT AN TOAN ----------
# Khong cho chay 9096 anh (~3 gio GPU) neu smoke test da cho ra output rac.
assert 'probe' in dir(), 'Chay o smoke test truoc da.'
_bad = sum(1 for r in probe if is_degenerate(r['raw']))
assert _bad == 0, (
    f'{_bad}/{len(probe)} anh o smoke test bi DEGENERATE (logits NaN, dtype={DTYPE}). '
    'Sua DTYPE_MODE roi chay lai smoke test truoc. Khong chay full de tranh dot GPU vo ich.')
assert any(r["text"] for r in probe), (
    'Smoke test khong doc ra chu nao. Xem chan doan o o tren truoc khi chay full.')
print('Smoke test da qua chot an toan -> bat dau chay full.\n')

records = []
ckpt_path = DEMO_OUTPUT / f'{TARGET_FOLDER}_ckpt.jsonl'

# --- resume: nap lai nhung gi da lam ---
done = set()
if ckpt_path.exists():
    with ckpt_path.open(encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue          # dong cuoi bi cat giua khi kernel bi kill
            records.append(r)
            done.add((r['video_id'], r['keyframe']))
    print(f'Resume: da co {len(done)} keyframe trong checkpoint, se bo qua.')

todo = [(vd, p) for vd, imgs in samples for p in imgs if (vd.name, p.name) not in done]
print(f'Con lai {len(todo)} / {total_keyframes} keyframe | {len(GPU_IDS)} worker\n')

lock = threading.Lock()
progress = {'n': 0, 'err': 0}
run_started = time.perf_counter()
ckpt_file = ckpt_path.open('a', encoding='utf-8')

def worker(gpu_id, chunk):
    torch.cuda.set_device(gpu_id)
    for video_dir, image_path in chunk:
        res = ocr_one(image_path, gpu_id=gpu_id)
        rec = {'video_id': video_dir.name, 'keyframe': image_path.name,
               'ms': round(res['total'] * 1000), 'gpu': gpu_id,
               'text': res['text'], 'raw': res['raw']}
        if res.get('error'):
            rec['error'] = res['error']
        with lock:
            records.append(rec)
            ckpt_file.write(json.dumps(rec, ensure_ascii=False) + '\n')
            progress['n'] += 1
            if res.get('error'):
                progress['err'] += 1
            n = progress['n']
            if n % CHECKPOINT_EVERY == 0 or n == len(todo):
                ckpt_file.flush()
                elapsed = time.perf_counter() - run_started
                rate = n / max(elapsed, 1e-9)
                print(f'[{n}/{len(todo)}] {rate:.2f} anh/s | ETA {(len(todo)-n)/max(rate,1e-9)/60:.1f} phut '
                      f'| loi {progress["err"]} | cuda:{gpu_id} {rec["video_id"]}/{rec["keyframe"]} '
                      f'-> {rec["text"][:55]}', flush=True)

try:
    threads = []
    for i, gid in enumerate(GPU_IDS):
        chunk = todo[i::len(GPU_IDS)]     # chia xen ke -> tai deu giua cac GPU
        if not chunk:
            continue
        t = threading.Thread(target=worker, args=(gid, chunk), daemon=True)
        t.start()
        threads.append(t)
    for t in threads:
        t.join()
finally:
    ckpt_file.flush()
    ckpt_file.close()

# thread chay xen ke -> sap xep lai cho on dinh
records.sort(key=lambda r: (r['video_id'], r['keyframe']))
wall = time.perf_counter() - run_started
print(f'\nXong {len(todo)} anh trong {wall/60:.1f} phut ({progress["err"]} loi). '
      f'Tong ban ghi: {len(records)}')

## Xuất báo cáo JSON

In [ ]:
import statistics

mean_ms  = statistics.fmean([r['ms'] for r in records]) if records else 0
with_text = [r for r in records if r['text']]
errored   = [r for r in records if r.get('error')]

report = {
    'model': MODEL_ID,
    'model_path': MODEL_PATH,
    'prompt': PROMPT,
    'res_mode': RES_MODE,
    'res_config': RES_CONFIG[RES_MODE],
    'gpus': [torch.cuda.get_device_name(i) for i in GPU_IDS],
    'n_workers': len(GPU_IDS),
    'dtype': str(DTYPE),
    'fp16_patch': bool(NEED_FP16_PATCH),
    'attn_implementation': 'eager',
    'max_length': MAX_LENGTH,
    'no_repeat_ngram_size': NO_REPEAT_NGRAM_SIZE,
    'ngram_window': NGRAM_WINDOW,
    'load_seconds': round(load_seconds, 1),
    'ms_per_keyframe': {'mean': round(mean_ms, 1)},
    'total_keyframes_processed': len(records),
    'keyframes_with_text': len(with_text),
    'keyframes_errored': len(errored),
    'results': [{'video_id': r['video_id'], 'keyframe': r['keyframe'],
                 'ms': r['ms'], 'text': r['text']} for r in records],
}
path = DEMO_OUTPUT / f'{TARGET_FOLDER}_unlimitedocr_report.json'
path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

print('Da luu:', path)
print(f'{len(with_text)}/{len(records)} keyframe co chu | trung binh {mean_ms:.0f} ms/anh '
      f'| {len(errored)} loi')
print(f'Ban tho (con nguyen tag <|det|> + toa do) nam trong checkpoint: {ckpt_path}')

In [ ]:
from IPython.display import FileLink
FileLink(str(path.relative_to('/kaggle/working')))

## Phụ lục A — dùng luôn hàm vẽ box của model

Nếu muốn ảnh có box do chính remote code render (kèm `result.md`), gọi `infer` với `save_results=True` và `eval_mode=False`. Lưu ý nó ghi tên file **cố định** (`result_with_boxes.jpg`, `result.md`) nên mỗi ảnh phải có `output_path` riêng, và nó **stream output ra stdout** (`eval_mode=False` thì `infer` trả về `None`).

```python
from IPython.display import Image as IPyImage, display

vd, p = flat[0]
outdir = DEMO_OUTPUT / 'viz' / p.stem
outdir.mkdir(parents=True, exist_ok=True)

with torch.cuda.device(GPU_IDS[0]):
    models[GPU_IDS[0]].infer(
        tokenizer, prompt=PROMPT, image_file=str(p), output_path=str(outdir),
        **RES_CONFIG[RES_MODE], max_length=MAX_LENGTH,
        no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE, ngram_window=NGRAM_WINDOW,
        temperature=TEMPERATURE,
        save_results=True, eval_mode=False,   # eval_mode PHAI False moi chay nhanh save_results
    )

display(IPyImage(filename=str(outdir / 'result_with_boxes.jpg'), width=900))
print((outdir / 'result.md').read_text(encoding='utf-8')[:1000])
```

## Phụ lục B — chạy bằng vLLM / SGLang (nhanh hơn nhiều)

`model.infer()` xử lý 1 ảnh/lần gọi và không batch được → với vài chục nghìn keyframe sẽ rất lâu. Nếu cần thông lượng thật, dựng server rồi gọi song song qua OpenAI-compatible API. Cả hai route **không cài chung env với các ô trên** — tách notebook riêng hoặc chạy ngoài Kaggle.

```bash
docker pull vllm/vllm-openai:unlimited-ocr          # CUDA 13.0
docker pull vllm/vllm-openai:unlimited-ocr-cu129    # Hopper, CUDA 12.9

python -m sglang.launch_server \
    --model baidu/Unlimited-OCR --served-model-name Unlimited-OCR \
    --attention-backend fa3 --page-size 1 --mem-fraction-static 0.8 \
    --context-length 32768 --host 0.0.0.0 --port 10000
```

Giữ nguyên tham số chống lặp của model card: `temperature=0`, `no_repeat_ngram_size=35`, `ngram_window=128` (ảnh đơn) / `1024` (nhiều trang).

## Phụ lục C — OCR nhiều trang / PDF bằng `infer_multi`

Không dùng cho keyframe (mỗi keyframe độc lập), để đây nếu cần OCR tài liệu. `infer_multi` **chỉ hỗ trợ chế độ `base`** và trả về tuple `(text, n_tokens)`:

```python
text, n_tokens = models[GPU_IDS[0]].infer_multi(
    tokenizer,
    prompt='<image>Multi page parsing.',
    image_files=['page1.png', 'page2.png', 'page3.png'],
    output_path='/kaggle/working/out',   # KHONG duoc de rong (os.makedirs vo dieu kien)
    image_size=1024,
    max_length=32768,                    # nhieu trang -> can max_length lon
    no_repeat_ngram_size=35, ngram_window=1024,
    save_results=True,
)
```

PDF thì convert sang ảnh trước bằng PyMuPDF (`fitz`), rồi truyền list đường dẫn vào `image_files`.